# ANEXO I. Semana 6: fine-tuning y despliegue

Notebook del curso SENA Iniciación a la IA para ADSO 2026. Las explicaciones completas están en la guía, anexo I.

Guarde su copia con **Archivo > Guardar una copia en Drive**.

Use GPU: **Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4**.

## I.1 Preparar los datos

Cree en Colab un archivo `datos.jsonl` con 40 líneas. Cada línea es una conversación en formato JSON:

```
{"messages": [{"role": "user", "content": "¿Qué es ADSO?"}, {"role": "assistant", "content": "ADSO es el programa tecnólogo del SENA en Análisis y Desarrollo de Software..."}]}
{"messages": [{"role": "user", "content": "¿Cómo creo una tarea en la API?"}, {"role": "assistant", "content": "Envía un POST a /tareas con un JSON que tenga el campo titulo..."}]}
```

Para crear el archivo desde una celda, escriba los ejemplos en una lista de Python y guárdelos así:

In [ ]:
import json

ejemplos = [
    ("¿Qué es ADSO?", "ADSO es el programa tecnólogo del SENA en Análisis y Desarrollo de Software..."),
    # agregue aquí sus 40 pares (pregunta, respuesta)
]

with open("datos.jsonl", "w", encoding="utf-8") as f:
    for pregunta, respuesta in ejemplos:
        f.write(json.dumps({"messages": [
            {"role": "user", "content": pregunta},
            {"role": "assistant", "content": respuesta}]}, ensure_ascii=False) + "\n")

Guarde aparte 10 preguntas que no estén en los datos, para la evaluación.

## I.2 Reentrenar con LoRA

Use Colab con GPU T4 (Anexo A.7).

In [ ]:
!pip install -q -U transformers trl peft datasets accelerate
!pip uninstall -q -y torchao

La segunda línea quita la librería `torchao` que trae Colab: su versión es más vieja que la que exige `peft` y sin quitarla el entrenamiento falla con `ImportError: Found an incompatible version of torchao`. También puede aparecer el mensaje rojo de `pip's dependency resolver` explicado en el Anexo E.1: ignórelo.

In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

BASE = "Qwen/Qwen2.5-0.5B-Instruct"
datos = load_dataset("json", data_files="datos.jsonl", split="train")
tokenizer = AutoTokenizer.from_pretrained(BASE)
# float32: la GPU T4 gratis no soporta bien el formato bfloat16
modelo = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32)

entrenador = SFTTrainer(
    model=modelo,
    processing_class=tokenizer,
    train_dataset=datos,
    args=SFTConfig(output_dir="modelo-ajustado", num_train_epochs=5,
                   per_device_train_batch_size=4, learning_rate=2e-4,
                   bf16=False, fp16=False,
                   logging_steps=5, save_strategy="no", report_to="none"),
    peft_config=LoraConfig(r=16, lora_alpha=32, target_modules="all-linear",
                           task_type="CAUSAL_LM"),
)
entrenador.train()

Con 40 ejemplos el entrenamiento tarda menos de un minuto en la T4 (más la descarga del modelo, de casi 1 GB). La pérdida (loss) debe bajar: en la prueba de esta guía pasó de 2,29 a 0,05.

Otras opciones de modelo base pequeño (cambie `BASE`): `HuggingFaceTB/SmolLM2-360M-Instruct` o `Qwen/Qwen2.5-1.5B-Instruct` (más lento).

## I.3 Comparar original y ajustado

In [ ]:
from transformers import pipeline

# Fusionar el ajuste LoRA con el modelo y pasarlo a modo de uso (no de entrenamiento).
# Sin estos pasos el modelo ajustado responde texto repetido sin sentido.
entrenador.model.eval()
modelo_ajustado = entrenador.model.merge_and_unload()
modelo_ajustado.config.use_cache = True

ajustado = pipeline("text-generation", model=modelo_ajustado, tokenizer=tokenizer)
original = pipeline("text-generation", model=BASE, dtype=torch.float32, device=0)


def preguntar(generador, pregunta):
    salida = generador([{"role": "user", "content": pregunta}], max_new_tokens=200)
    return salida[0]["generated_text"][-1]["content"]


preguntas_prueba = [
    "Escriba aquí una pregunta que NO esté en los datos",
    # ... sus 10 preguntas
]

for p in preguntas_prueba:
    print("PREGUNTA:", p)
    print("ORIGINAL:", preguntar(original, p))
    print("AJUSTADO:", preguntar(ajustado, p))
    print("-" * 60)

Llene la tabla del Anexo H.4 con estos resultados. Fíjese en los errores del modelo ajustado: en las pruebas de esta guía respondió con seguridad cosas falsas que no estaban en los datos, por ejemplo que la API usa "Python con el módulo Flask" (usa FastAPI) o que una tarea se elimina con un POST. Pregúntese: ¿esto se habría resuelto mejor con RAG (semana 4)?

## I.4 Publicar el modelo

Se publica el modelo ya fusionado en el paso I.3. Sube unos 2 GB y tarda alrededor de un minuto:

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))
modelo_ajustado.push_to_hub("modelo-ajustado-sena")
tokenizer.push_to_hub("modelo-ajustado-sena")

## I.5 Aplicación web con Gradio

La aplicación corre en Colab (no necesita GPU) y Gradio le da un enlace público para abrirla desde el celular o compartirla.

> Hugging Face ya no permite crear aplicaciones con Gradio (Spaces) en cuentas gratis: solo en el plan PRO. Por eso la aplicación se ejecuta en Colab. Si usted tiene PRO, puede subir este mismo código a un Space.

Abra un notebook nuevo en Colab y ejecute:

In [ ]:
!pip install -q gradio

Luego, en otra celda (cambie `SU_USUARIO` por su usuario de Hugging Face):

In [ ]:
import gradio as gr
import torch
from transformers import pipeline

generador = pipeline("text-generation", model="SU_USUARIO/modelo-ajustado-sena",
                     dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                     device_map="auto")


def texto(contenido):
    # Gradio puede entregar el mensaje como texto o como lista de partes
    if isinstance(contenido, list):
        return " ".join(p.get("text", "") for p in contenido if isinstance(p, dict))
    return str(contenido)


def responder(mensaje, historial):
    mensajes = [{"role": m["role"], "content": texto(m["content"])} for m in historial]
    mensajes.append({"role": "user", "content": texto(mensaje)})
    salida = generador(mensajes, max_new_tokens=200)
    return salida[0]["generated_text"][-1]["content"]


demo = gr.ChatInterface(responder, title="Asistente ajustado - SENA")
demo.launch(share=True)

1. El chat aparece debajo de la celda y también en el enlace `https://....gradio.live` que muestra la salida. Ese enlace funciona mientras el notebook esté abierto (Colab apaga el entorno después de un tiempo sin uso) y dura máximo una semana.

2. Si Colab pregunta si el notebook puede usar el secreto `HF_TOKEN`, puede responder Cancelar: el modelo es público y se descarga sin token.

3. Para la evidencia, grabe un video corto o tome capturas del chat respondiendo con el enlace público visible, y comparta el notebook.

> En la prueba de esta guía, con CPU, el modelo se descargó en menos de un minuto y cada respuesta tardó unos segundos. El chat recuerda la conversación: a "¿Cómo listo las tareas?" respondió "Haz un GET a /tareas." y luego a "¿Y qué pasa si no existe?" respondió "Devuelve un 404."